## sanity check

In [2]:
import pycalphad
print(dir(pycalphad))
print(pycalphad.__file__)

['CalculateError', 'ConditionError', 'Database', 'DofError', 'EquilibriumError', 'Model', 'ReferenceState', 'Workspace', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_discovered_plugins', '_finder', '_ispkg', '_name', 'as_property', 'binplot', 'calculate', 'codegen', 'core', 'eqplot', 'equilibrium', 'importlib', 'io', 'mapping', 'model', 'pkgutil', 'plot', 'property_framework', 'pycalphad', 'ternplot', 'v', 'variables', 'warnings']
/run/media/adi/New Volume/conda-envs/struct/lib/python3.11/site-packages/pycalphad/__init__.py


## database check

In [5]:
from pycalphad import Database
from tinydb import where
from itertools import combinations

TARGET = {'FE', 'NI', 'CO', 'MO', 'TI', 'AL'}

def check_db(path, name):
    db = Database(path)
    db_elements = db.elements - {'VA', '/-'}

    print(f"\n{'='*55}")
    print(f"DATABASE: {name}")
    print(f"{'='*55}")
    print(f"All elements : {sorted(db_elements)}")
    print(f"Missing (ours): {sorted(TARGET - db_elements) or 'None'}")

    interactions = set()
    for phase_name in db.phases.keys():
        for param in db._parameters.search(where('phase_name') == phase_name):
            species = set()
            for subl in param['constituent_array']:
                for sp in subl:
                    el = str(sp).split(':')[0].upper()
                    if el != 'VA':
                        species.add(el)
            if len(species) >= 1:
                interactions.add(frozenset(species))

    for n, label in [(1,'UNARY'), (2,'BINARY'), (3,'TERNARY')]:
        hits = sorted([tuple(sorted(x)) for x in interactions
                       if len(x) == n])
        ours = [i for i in hits if set(i).issubset(TARGET)]
        print(f"\n--- {label} ({len(hits)} total, {len(ours)} relevant) ---")
        for i in hits:
            tag = " [OUR]" if set(i).issubset(TARGET) else ""
            print(f"  {i}{tag}")

    # coverage summary
    found = set(frozenset(x) for x in interactions)
    for n, label in [(2,'BINARY'), (3,'TERNARY')]:
        needed = [frozenset(p) for p in combinations(TARGET, n)]
        missing = sorted([tuple(sorted(x)) for x in needed if x not in found])
        print(f"\nMissing {label}: {missing or 'None'}")

    print(f"\nPhases: {sorted(db.phases.keys())}")

# run on both
check_db("../databases/COST507-modified.tdb", "COST507")
check_db("../databases/Fe-Ni-Ti_DeKeyzer2009.tdb", "Fe-Ni-Ti_DeKeyzer2009")
check_db("../databases/mc_fecocrnbti.tdb", "mc_fecocrnbti")


DATABASE: COST507
All elements : ['AL', 'AR', 'B', 'C', 'CE', 'CR', 'CU', 'FE', 'HF', 'LI', 'MG', 'MN', 'MO', 'N', 'NB', 'ND', 'NI', 'O', 'SI', 'SN', 'TA', 'TI', 'V', 'W', 'Y', 'ZN', 'ZR']
Missing (ours): ['CO']

--- UNARY (50 total, 5 relevant) ---
  ('AL',) [OUR]
  ('AL1',)
  ('AL2',)
  ('B',)
  ('B1',)
  ('B1C1',)
  ('B1N1',)
  ('B2',)
  ('C',)
  ('C1',)
  ('C1+1',)
  ('C1-1',)
  ('C1SI1',)
  ('C1SI2',)
  ('C2',)
  ('C2-1',)
  ('C2SI1',)
  ('C3',)
  ('C4',)
  ('C5',)
  ('CE',)
  ('CR',)
  ('CU',)
  ('FE',) [OUR]
  ('HF',)
  ('LI',)
  ('MG',)
  ('MN',)
  ('MO',) [OUR]
  ('N',)
  ('N1',)
  ('N2',)
  ('N3',)
  ('NB',)
  ('ND',)
  ('NI',) [OUR]
  ('SI',)
  ('SI+1',)
  ('SI1',)
  ('SI2',)
  ('SI3',)
  ('SN',)
  ('TA',)
  ('TI',) [OUR]
  ('TI1',)
  ('V',)
  ('W',)
  ('Y',)
  ('ZN',)
  ('ZR',)

--- BINARY (87 total, 5 relevant) ---
  ('AL', 'B')
  ('AL', 'C')
  ('AL', 'CE')
  ('AL', 'CR')
  ('AL', 'CU')
  ('AL', 'FE') [OUR]
  ('AL', 'LI')
  ('AL', 'MG')
  ('AL', 'MN')
  ('AL', 'MO') [OUR]

/run/media/adi/New Volume/conda-envs/struct/lib/python3.11/site-packages/pycalphad/io/tdb.py:995: UserWarning: The type definition character `%` was defined in the following phases: ['LIQUID', 'A1', 'FCC4', 'A2', 'BCC2', 'A3', 'C14', 'NI3TI', 'NITI2', 'C15', 'C36'], but no corresponding TYPE_DEFINITION line was found in the TDB.
  warnings.warn(f"The type definition character `{typechar}` was defined in the following phases: "
/run/media/adi/New Volume/conda-envs/struct/lib/python3.11/site-packages/pycalphad/io/tdb.py:995: UserWarning: The type definition character `B` was defined in the following phases: ['FCC4', 'BCC2'], but no corresponding TYPE_DEFINITION line was found in the TDB.
  warnings.warn(f"The type definition character `{typechar}` was defined in the following phases: "



DATABASE: Fe-Ni-Ti_DeKeyzer2009
All elements : ['FE', 'NI', 'TI']
Missing (ours): ['AL', 'CO', 'MO']

--- UNARY (3 total, 3 relevant) ---
  ('FE',) [OUR]
  ('NI',) [OUR]
  ('TI',) [OUR]

--- BINARY (3 total, 3 relevant) ---
  ('FE', 'NI') [OUR]
  ('FE', 'TI') [OUR]
  ('NI', 'TI') [OUR]

--- TERNARY (2 total, 1 relevant) ---
  ('*', 'NI', 'TI')
  ('FE', 'NI', 'TI') [OUR]

Missing BINARY: [('AL', 'CO'), ('AL', 'FE'), ('AL', 'MO'), ('AL', 'NI'), ('AL', 'TI'), ('CO', 'FE'), ('CO', 'MO'), ('CO', 'NI'), ('CO', 'TI'), ('FE', 'MO'), ('MO', 'NI'), ('MO', 'TI')]

Missing TERNARY: [('AL', 'CO', 'FE'), ('AL', 'CO', 'MO'), ('AL', 'CO', 'NI'), ('AL', 'CO', 'TI'), ('AL', 'FE', 'MO'), ('AL', 'FE', 'NI'), ('AL', 'FE', 'TI'), ('AL', 'MO', 'NI'), ('AL', 'MO', 'TI'), ('AL', 'NI', 'TI'), ('CO', 'FE', 'MO'), ('CO', 'FE', 'NI'), ('CO', 'FE', 'TI'), ('CO', 'MO', 'NI'), ('CO', 'MO', 'TI'), ('CO', 'NI', 'TI'), ('FE', 'MO', 'NI'), ('FE', 'MO', 'TI'), ('MO', 'NI', 'TI')]

Phases: ['A1', 'A2', 'A3', 'BCC2', 'C14'

# Database Selection Verdict

## Selected: `Fe-Ni-Ti_DeKeyzer2009.tdb`

### Why this database
- Only database among the three that explicitly defines the **NI3TI phase** - the central precipitate of this entire study
- Contains all three required unary descriptions: **FE, NI, TI**
- Contains all three relevant binaries: **FE-NI, FE-TI, NI-TI**
- Contains the **FE-NI-TI ternary** interaction — essential for realistic phase fraction predictions in a multicomponent Fe-Ni martensite matrix
- Also defines NITI2, C14/C15/C36 Laves phases, and the LIQUID phase — giving a reasonably complete picture of competing phases in the Fe-Ni-Ti system
- Specifically assessed and published for the Fe-Ni-Ti system (DeKeyzer et al., 2009), meaning its Gibbs energy parameters were fitted to Fe-Ni-Ti experimental data — not borrowed from an unrelated alloy family

### Why the others were rejected

| Database | Fatal flaw |
|---|---|
| `COST507` | Designed for aluminium alloys. Has FE, NI, TI unaries and some binaries but **no NI3TI phase defined**, no FE-NI binary, no NI-TI binary. Cannot model the precipitate of interest at all. |
| `mc_fecocrnbti` | Has CO and FE-TI but **no NI element**, no NI-TI binary, no NI3TI phase. Structurally incapable of describing Ni₃Ti precipitation regardless of composition input. |

---

## Known Limitations of the Selected Database

### 1. Mo is absent
The alloy contains **5 wt% Mo** — a major alloying element. Mo-enriched particles are the heterogeneous nucleation precursors for Ni₃Ti in the DAT sequence (the core mechanism of Xu et al. 2025, Section 4.1). This database cannot model:
- Mo precipitation from the BCC matrix during/after SST
- Mo-enriched particle growth during DAT-1 (370°C)
- Mo segregation to the Ni₃Ti/matrix interface during DAT-2
- Fe₇Mo₂ (ω phase) formation

**Mitigation:** The Mo-related sequence is discussed qualitatively citing Xu et al. Section 4.1. DFT formation energies of Fe₇Mo₂ are cross-validated separately via the Materials Project API.

### 2. Co is absent
The alloy contains **8.5 wt% Co**. Per Xu et al. ref [47], cobalt explicitly reduces the solubility of Mo in maraging steel, which drives Mo precipitation — part of the DAT-1 mechanism. This effect cannot be captured computationally.

**Mitigation:** Acknowledged as a limitation. The Co effect on Mo solubility is discussed qualitatively.

### 3. Al is absent
The alloy contains **0.2 wt% Al**. Al is a minor addition in this composition and its primary role is deoxidation during melting. Its thermodynamic effect on Ni₃Ti stability is expected to be small at this concentration.

**Mitigation:** Negligible impact on the primary validation targets. Noted in limitations section.

### 4. Reduced system — composition must be rescaled
All CALPHAD calculations are performed on the **Fe-Ni-Ti ternary subsystem** only, with the alloy composition renormalized to exclude Co, Mo, and Al. This simplification affects absolute phase fraction values and transformation temperatures but preserves the qualitative precipitation sequence.

**Mitigation:** Discrepancies between computed and experimental phase fractions are expected and are explicitly discussed. The model is used for trend validation, not quantitative replication.

### 5. Non-critical TYPE_DEFINITION warnings on load
The database triggers two `UserWarning` messages about missing `TYPE_DEFINITION` lines for characters `%` and `B` in several phases. These are a known formatting quirk of older TDB files. pycalphad parses the thermodynamic parameters correctly regardless — confirmed by successful phase identification in test calculations.

---

## Summary